# Applied Data Science Capstone — 6. Interactive Dashboard (Plotly Dash)

## Objective
Build the course's SpaceX launch dashboard as a Plotly Dash app:

* **Pie chart** — successful launches by site (or success vs. failure for one site),
* **Payload range slider** — filters the scatter plot,
* **Scatter chart** — payload vs. landing success, coloured by booster version.

The app code below is complete and runnable (`python app.py`);
the server is *not* started inside the notebook.
The two component figures are rendered here statically for inspection.

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/dataset_spacex.csv")
print(df.shape)
print(df["LaunchSite"].unique())

(90, 23)
<StringArray>
['CCAFS LC-40', 'CCAFS SLC-40', 'KSC LC-39A', 'VAFB SLC-4E']
Length: 4, dtype: str


### Component 1 — pie chart: successful launches by site

In [2]:
success_by_site = (df[df["Class"] == 1]
                   .groupby("LaunchSite").size().reset_index(name="successes"))
fig_pie = px.pie(success_by_site, values="successes", names="LaunchSite",
                 title="Successful first-stage landings by launch site")
fig_pie

### Component 2 — scatter: payload vs. landing success

In [3]:
fig_sc = px.scatter(df, x="PayloadMass", y="Class",
                    color="BoosterVersion", size="ReusedCount",
                    hover_data=["LaunchSite", "Orbit", "Date"],
                    title="Payload mass vs. landing success (colour = booster version)",
                    labels={"Class": "Landed (1=yes, 0=no)"})
fig_sc

### Full Dash application
Dropdown selects a site (or ALL); the pie chart updates to success/failure
share for that site; the payload slider filters the scatter; the scatter
colours points by booster version category.

In [4]:
# Save as app.py next to the notebooks and run with:  python app.py
dash_app_code = r"""
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

df = pd.read_csv("../data/dataset_spacex.csv")

app = Dash(__name__)
app.title = "SpaceX Falcon 9 — Landing Prediction Dashboard"

app.layout = html.Div([
    html.H1("SpaceX Falcon 9 First-Stage Landing Dashboard"),
    dcc.Dropdown(
        id="site-dropdown",
        options=[{"label": "All sites", "value": "ALL"}] +
                [{"label": s, "value": s} for s in sorted(df["LaunchSite"].unique())],
        value="ALL", clearable=False,
    ),
    dcc.Graph(id="success-pie"),
    html.P("Payload range (kg):"),
    dcc.RangeSlider(id="payload-slider",
                    min=int(df["PayloadMass"].min()), max=int(df["PayloadMass"].max()),
                    step=500, value=[int(df["PayloadMass"].min()),
                                     int(df["PayloadMass"].max())]),
    dcc.Graph(id="success-scatter"),
])

@app.callback(Output("success-pie", "figure"), Input("site-dropdown", "value"))
def update_pie(site):
    d = df if site == "ALL" else df[df["LaunchSite"] == site]
    if site == "ALL":
        g = d[d["Class"] == 1].groupby("LaunchSite").size().reset_index(name="n")
        return px.pie(g, values="n", names="LaunchSite",
                      title="Successful landings by site")
    g = d.groupby("Class").size().reset_index(name="n")
    return px.pie(g, values="n", names="Class",
                  title=f"Landing outcome — {site} (1=landed, 0=lost)")

@app.callback(Output("success-scatter", "figure"),
              [Input("site-dropdown", "value"), Input("payload-slider", "value")])
def update_scatter(site, prange):
    d = df[(df["PayloadMass"] >= prange[0]) & (df["PayloadMass"] <= prange[1])]
    if site != "ALL":
        d = d[d["LaunchSite"] == site]
    return px.scatter(d, x="PayloadMass", y="Class", color="BoosterVersion",
                      hover_data=["LaunchSite", "Orbit"],
                      title="Payload vs. landing success")

if __name__ == "__main__":
    app.run_server(debug=False)
"""

with open("../notebooks/spacex_dash_app.py", "w") as f:
    f.write(dash_app_code.strip() + "\n")
print("saved ../notebooks/spacex_dash_app.py")
print()
print(dash_app_code[:400], "...")

saved ../notebooks/spacex_dash_app.py


import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

df = pd.read_csv("../data/dataset_spacex.csv")

app = Dash(__name__)
app.title = "SpaceX Falcon 9 — Landing Prediction Dashboard"

app.layout = html.Div([
    html.H1("SpaceX Falcon 9 First-Stage Landing Dashboard"),
    dcc.Dropdown(
        id="site-dropdown",
        options=[{"label": "All sites", ...


### How to run
```bash
cd notebooks
pip install dash plotly pandas
python spacex_dash_app.py
# open http://127.0.0.1:8050
```